# PMIL — Storage-Frugal, Algorithm-Faithful Colab Pipeline (v2)

Reproduces the actual pipeline in your uploaded repo: **Max-MIL training -> patch feature extraction ->
Affinity Propagation prototype discovery -> PMIL training/testing** — while never holding more than one
slide's raw data on local disk.

**v2 changes from the first draft:**
- Real streaming **Max-MIL training** (top-k instance selection, re-scored every epoch), not frozen
  ImageNet features. This is the part that actually reproduces the paper's method.
- Fixed a coordinate bug: tile coordinates are now always stored/read in **level-0 pixel space**, as
  `OpenSlide.read_region` requires, regardless of which pyramid `level` you tile at.
- Uses `lib['targets']` (plural) to match what `max-mil.py`/`pmil.py` actually read from the `.ckpt` files
  (the README's prose says singular `'target'`, but the code that runs uses plural — verified directly
  against your uploaded `max-mil.py` and `pmil.py`).

**The trade-off this creates:** faithful Max-MIL needs to re-score every patch of every slide once per
epoch. To keep storage flat, each epoch **re-downloads every slide** (briefly, one at a time) rather than
keeping patches resident. Total network/time cost scales as `maxmil_nepochs x num_slides`, so start with a
small `maxmil_nepochs` (default 5) and a small manifest, and raise both once you've confirmed it works.

**You still need to provide `manifest.csv`** (`slide_name,url,label,split`) with real, license-compliant
download links (Camelyon16: https://camelyon17.grand-challenge.org/Data/). I can't hardcode working bulk
URLs for you.


In [ ]:
# 1. Mount Drive & set up folders
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/PMIL_run'
SCRATCH   = '/content/scratch'   # local disk -- never holds more than 1 slide at a time

for d in ['lib', 'feat', 'model', 'cluster', 'result']:
    os.makedirs(os.path.join(BASE_DIR, d), exist_ok=True)
os.makedirs(SCRATCH, exist_ok=True)
print('Drive project dir:', BASE_DIR)


In [ ]:
# 2. GPU check -- do this before touching any WSIs
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Go to Runtime -> Change runtime type -> GPU before continuing.')


In [ ]:
# 3. Install dependencies
!apt-get -qq update && apt-get -qq install -y openslide-tools > /dev/null
!pip -q install openslide-python gdown scikit-learn pandas pillow requests


In [ ]:
# 4. Config (replaces the original argparse CLI -- notebooks don't get argv)
import shutil as _shutil

class Config:
    manifest_csv   = os.path.join(BASE_DIR, 'manifest.csv')   # <-- you supply this

    # tiling
    tile_size      = 256
    level          = 0            # OpenSlide pyramid level to tile at
    tissue_thresh  = 0.10
    max_tiles_per_slide = 4000     # safety cap; raise once you trust your setup

    # Max-MIL (streaming, faithful reproduction)
    maxmil_nepochs = 5             # << each epoch re-downloads every slide once. Start small.
    maxmil_k       = 1             # top-k tiles assumed positive-class per slide (1 = standard MIL)
    maxmil_lr      = 1e-4
    maxmil_batch   = 128
    test_every     = 1
    patience       = 3

    # feature extraction (after Max-MIL is trained)
    feat_format    = '.npy'
    infer_batch    = 256

    # clustering (Affinity Propagation, as in cluster.py)
    preference     = 0
    damping        = 0.5
    lamb           = 0.25

    # PMIL training
    nepochs        = 30
    lr             = 1e-4
    weights        = 0.5

    # paths
    lib_dir        = os.path.join(BASE_DIR, 'lib')
    feat_dir       = os.path.join(BASE_DIR, 'feat')
    model_dir      = os.path.join(BASE_DIR, 'model')
    cluster_dir    = os.path.join(BASE_DIR, 'cluster')
    result_dir     = os.path.join(BASE_DIR, 'result')
    mil_model      = os.path.join(model_dir, 'maxmil.pth')
    pmil_model     = os.path.join(model_dir, 'pmil_model.pth')
    global_cluster = os.path.join(cluster_dir, 'prototypes_features.npz')

    min_free_gb    = 2.0   # abort a slide's download if free scratch space drops below this

cfg = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def free_scratch_gb():
    total, used, free = _shutil.disk_usage(SCRATCH)
    return free / 1024**3

print('Device:', device, '| free scratch space:', round(free_scratch_gb(), 1), 'GB')


In [ ]:
# 5. Model -- resnet34 matching model/model_maxmil.py's architecture
#    (BasicBlock ResNet, forward returns (fc(x), pooled_features)); used both as the Max-MIL
#    instance classifier and, unchanged, as the IClassifier backbone that pmil.py wraps.
import torch.nn as nn
import torch.nn.functional as F
from torch.hub import load_state_dict_from_url

model_urls = {'resnet34': 'https://download.pytorch.org/models/resnet34-333f7ec4.pth'}

def conv3x3(i, o, stride=1): return nn.Conv2d(i, o, 3, stride, 1, bias=False)
def conv1x1(i, o, stride=1): return nn.Conv2d(i, o, 1, stride, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1, self.bn1 = conv3x3(inplanes, planes, stride), nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2, self.bn2 = conv3x3(planes, planes), nn.BatchNorm2d(planes)
        self.downsample, self.stride = downsample, stride

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet34(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._make_layer(64, 3)
        self.layer2 = self._make_layer(128, 4, stride=2)
        self.layer3 = self._make_layer(256, 6, stride=2)
        self.layer4 = self._make_layer(512, 3, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(conv1x1(self.inplanes, planes, stride), nn.BatchNorm2d(planes))
        layers = [BasicBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes
        layers += [BasicBlock(self.inplanes, planes) for _ in range(1, blocks)]
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x).reshape(x.size(0), -1)
        return self.fc(x), x   # (logits, 512-d pooled feature) -- matches model_maxmil.resnet34


def build_resnet34(pretrained_imagenet=True):
    model = ResNet34()
    if pretrained_imagenet:
        sd = load_state_dict_from_url(model_urls['resnet34'], progress=True)
        model.load_state_dict(sd, strict=False)   # fc shape differs (2 vs 1000), loaded partially
    return model


In [ ]:
# 6. PMIL-side models (from model/model_pmil.py), wrapping the resnet34 above unchanged
class IClassifier(nn.Module):
    def __init__(self, mil_model_path):
        super().__init__()
        temp = build_resnet34(pretrained_imagenet=False)
        ch = torch.load(mil_model_path, map_location='cpu')
        temp.load_state_dict(ch['state_dict'])
        self.features = nn.Sequential(temp.conv1, temp.bn1, temp.relu, temp.maxpool,
                                       temp.layer1, temp.layer2, temp.layer3, temp.layer4, temp.avgpool)
        self.fc = temp.fc

    def forward(self, features):
        # `features` here are already-pooled 512-d vectors (as in the original IClassifier.forward)
        return features, self.fc(features)


class BClassifier(nn.Module):
    def __init__(self, num_cluster, feature_size=512, input_size=128, output_class=2, dropout=0):
        super().__init__()
        self.transformer = nn.Linear(feature_size, input_size)
        self.classifier = nn.Sequential(
            nn.Linear(num_cluster, 128), nn.ReLU(True),
            nn.Dropout(dropout), nn.Linear(128, output_class))

    def forward(self, features, probs, prototype):
        _, m_indices = torch.sort(probs, 0, descending=True)
        k = min(5, features.size(0))
        m_features = torch.index_select(features, 0, m_indices[:k, 1])
        f, p = self.transformer(m_features), self.transformer(prototype)
        sim = euclidean_similarity_matrix(f, p)
        cmax, _ = torch.max(sim, dim=1, keepdim=True)
        sim = sim / cmax
        coding = torch.mean(sim, dim=0, keepdim=True)
        return self.classifier(coding), coding


class PBMIL(nn.Module):
    def __init__(self, i_classifier, b_classifier):
        super().__init__()
        self.i_classifier, self.b_classifier = i_classifier, b_classifier

    def forward(self, x, prototype):
        features, score = self.i_classifier(x)
        probs = F.softmax(score, dim=1)
        output, coding = self.b_classifier(features, probs, prototype)
        return score, output, coding


def euclidean_similarity_matrix(a, b):
    out = torch.zeros(a.size(0), b.size(0), device=a.device)
    for i in range(a.size(0)):
        out[i, :] = torch.pairwise_distance(a[i, :], b)
    return out


class EarlyStopping:
    def __init__(self, model_path, patience=7, verbose=False):
        self.patience, self.verbose = patience, verbose
        self.counter, self.best_score, self.early_stop = 0, None, False
        self.model_path = model_path

    def __call__(self, val_loss, state_dict, extra=None):
        score = -val_loss
        if self.best_score is None or score > self.best_score:
            self.best_score, self.counter = score, 0
            obj = {'state_dict': state_dict}
            if extra: obj.update(extra)
            torch.save(obj, self.model_path)
        else:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True


In [ ]:
# 7. WSI tiling -- coordinates are ALWAYS returned/consumed in level-0 pixel space
#    (fixes the earlier bug: OpenSlide.read_region always expects level-0 coords, even when
#    reading a downsampled `level`).
import numpy as np
import openslide
from PIL import Image

def tissue_grid(slide_path, tile_size=256, level=0, tissue_thresh=0.10, max_tiles=4000):
    slide = openslide.OpenSlide(slide_path)
    downsample = slide.level_downsamples[level]
    w, h = slide.level_dimensions[level]              # dims in `level` space
    thumb = slide.get_thumbnail((max(w // 32, 1), max(h // 32, 1))).convert('HSV')
    sat = np.array(thumb)[:, :, 1]
    thresh = max(sat.mean() * 0.5, 20)
    tissue_mask = sat > thresh
    scale_x = tissue_mask.shape[1] / w
    scale_y = tissue_mask.shape[0] / h

    coords = []
    for y in range(0, h - tile_size + 1, tile_size):
        for x in range(0, w - tile_size + 1, tile_size):
            mx0, mx1 = int(x * scale_x), max(int((x + tile_size) * scale_x), int(x * scale_x) + 1)
            my0, my1 = int(y * scale_y), max(int((y + tile_size) * scale_y), int(y * scale_y) + 1)
            if tissue_mask[my0:my1, mx0:mx1].mean() >= tissue_thresh:
                # convert level-space (x,y) -> level-0 space for storage & read_region
                coords.append((int(x * downsample), int(y * downsample)))
            if len(coords) >= max_tiles:
                slide.close()
                return coords
    slide.close()
    return coords


def read_tile(slide, coord0, tile_size, level):
    """coord0 must already be in level-0 pixel space."""
    return slide.read_region(coord0, level, (tile_size, tile_size)).convert('RGB')


## Sanity check: look at actual tiles before running anything expensive

Run this on ONE slide from your manifest first. A tiling/tissue-detection pipeline can run
with zero errors while quietly extracting background instead of tissue -- catch that here,
not after a multi-epoch Max-MIL run.


In [ ]:
# Validation cell -- visualize a sample of extracted tiles for one slide
import matplotlib.pyplot as plt

def preview_slide_tiles(name, url, n_preview=12):
    coords = ensure_grid(name, url)
    print(f'{name}: {len(coords)} tissue tiles found')
    if not coords:
        return
    ext = '.tif' if not url.startswith('gdrive:') else '.svs'
    raw_path = os.path.join(SCRATCH, name + ext)
    try:
        download_slide(url, raw_path)
        slide = openslide.OpenSlide(raw_path)
        sample = coords[:: max(1, len(coords) // n_preview)][:n_preview]
        fig, axes = plt.subplots(1, len(sample), figsize=(2.2 * len(sample), 2.5))
        if len(sample) == 1:
            axes = [axes]
        for ax, coord in zip(axes, sample):
            ax.imshow(read_tile(slide, coord, cfg.tile_size, cfg.level))
            ax.set_title(f'{coord}', fontsize=8)
            ax.axis('off')
        plt.tight_layout(); plt.show()
        slide.close()
    finally:
        if os.path.exists(raw_path):
            os.remove(raw_path)

# Example -- replace with a real row from your manifest before running:
# rows = load_manifest(cfg)
# preview_slide_tiles(rows[0]['slide_name'], rows[0]['url'])


In [ ]:
# 8. Streamed download, one slide at a time
import requests

def download_slide(url, dest_path, chunk_size=8 * 1024 * 1024):
    if url.startswith('gdrive:'):
        import gdown
        gdown.download(id=url.split('gdrive:', 1)[1], output=dest_path, quiet=False)
        return dest_path
    if free_scratch_gb() < cfg.min_free_gb:
        raise RuntimeError(f'Only {free_scratch_gb():.1f} GB free on scratch disk -- aborting download.')
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest_path


In [ ]:
# 9. Grid cache -- tile coordinates only (tiny CSVs), computed once per slide, reused every
#    Max-MIL epoch and by the final feature-extraction pass, so tissue detection isn't repeated.
import csv

def grid_path_for(name):
    return os.path.join(cfg.feat_dir, name + '.grid.csv')

def ensure_grid(name, url):
    gp = grid_path_for(name)
    if os.path.exists(gp):
        return [tuple(map(int, l.strip().split(','))) for l in open(gp) if l.strip()]
    ext = '.tif' if not url.startswith('gdrive:') else '.svs'
    raw_path = os.path.join(SCRATCH, name + ext)
    try:
        download_slide(url, raw_path)
        coords = tissue_grid(raw_path, cfg.tile_size, cfg.level, cfg.tissue_thresh, cfg.max_tiles_per_slide)
        with open(gp, 'w') as f:
            f.write('\n'.join(f'{x},{y}' for x, y in coords))
        return coords
    finally:
        if os.path.exists(raw_path):
            os.remove(raw_path)


def load_manifest(cfg):
    rows = list(csv.DictReader(open(cfg.manifest_csv)))
    for r in rows:
        r['label'] = int(r['label'])
    return rows


In [ ]:
# 10. Streaming Max-MIL training (faithful to max-mil.py's algorithm):
#     every epoch, re-score ALL patches of ALL train slides with the CURRENT model, keep only the
#     top-k highest-scoring patch(es) per slide (standard-MIL "positive evidence" assumption), and
#     train one step on just that small selected set. Each slide is downloaded once per epoch and
#     deleted immediately after scoring -- storage never holds more than one slide.
import torch.optim as optim
from torch.optim.lr_scheduler import MultiStepLR
import torchvision.transforms as transforms

normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.1, 0.1, 0.1])
to_tensor = transforms.Compose([transforms.ToTensor(), normalize])


def score_slide_and_collect_topk(name, url, model, k, batch_size):
    """Downloads the slide once, scores every tissue patch with `model`, returns the top-k
    patch tensors (positive-class prob highest) plus the slide's own max prob (for eval)."""
    coords = ensure_grid(name, url)
    if not coords:
        return torch.empty(0, 3, cfg.tile_size, cfg.tile_size), None
    ext = '.tif' if not url.startswith('gdrive:') else '.svs'
    raw_path = os.path.join(SCRATCH, name + ext)
    try:
        download_slide(url, raw_path)
        slide = openslide.OpenSlide(raw_path)
        best_probs = torch.full((len(coords),), -1.0)
        tiles_cache = {}   # only ever holds the running top-k tensors, not all patches
        buf_imgs, buf_idx = [], []

        def flush():
            if not buf_imgs:
                return
            x = torch.stack(buf_imgs).to(device)
            with torch.no_grad():
                logits, _ = model(x)
                p = F.softmax(logits, dim=1)[:, 1].cpu()
            for j, idx in enumerate(buf_idx):
                best_probs[idx] = p[j]
                tiles_cache[idx] = buf_imgs[j]
            buf_imgs.clear(); buf_idx.clear()

        for i, coord in enumerate(coords):
            img = read_tile(slide, coord, cfg.tile_size, cfg.level)
            buf_imgs.append(to_tensor(img)); buf_idx.append(i)
            if len(buf_imgs) == batch_size:
                flush()
        flush()
        slide.close()

        topk_idx = torch.topk(best_probs, k=min(k, len(coords))).indices.tolist()
        topk_tensors = torch.stack([tiles_cache[i] for i in topk_idx])
        slide_max_prob = float(best_probs.max())
        return topk_tensors, slide_max_prob
    finally:
        if os.path.exists(raw_path):
            os.remove(raw_path)


def run_maxmil_training(cfg):
    rows = load_manifest(cfg)
    train_rows = [r for r in rows if r['split'] == 'train']
    val_rows   = [r for r in rows if r['split'] == 'val']

    model = build_resnet34(pretrained_imagenet=True).to(device)
    criterion = nn.CrossEntropyLoss().to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.maxmil_lr, weight_decay=1e-4)
    scheduler = MultiStepLR(optimizer, milestones=[int(cfg.maxmil_nepochs*0.6)], gamma=0.1)
    early_stop = EarlyStopping(cfg.mil_model, patience=cfg.patience, verbose=True)

    for epoch in range(cfg.maxmil_nepochs):
        model.eval()
        batch_x, batch_y = [], []
        for row in train_rows:
            topk_tensors, _ = score_slide_and_collect_topk(row['slide_name'], row['url'], model,
                                                            cfg.maxmil_k, cfg.infer_batch)
            for t in topk_tensors:
                batch_x.append(t); batch_y.append(row['label'])

        # train one epoch on the selected top-k instances from every slide
        model.train()
        perm = torch.randperm(len(batch_x))
        running_loss = 0.
        for s in range(0, len(perm), cfg.maxmil_batch):
            idx = perm[s:s+cfg.maxmil_batch]
            x = torch.stack([batch_x[i] for i in idx]).to(device)
            y = torch.tensor([batch_y[i] for i in idx]).to(device)
            logits, _ = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            running_loss += loss.item() * len(idx)
        scheduler.step()
        train_loss = running_loss / max(len(perm), 1)
        print(f'[Max-MIL] epoch {epoch+1}/{cfg.maxmil_nepochs}  train_loss={train_loss:.4f}')

        if val_rows and (epoch + 1) % cfg.test_every == 0:
            model.eval()
            val_max_probs, val_targets = [], []
            for row in val_rows:
                _, slide_max_prob = score_slide_and_collect_topk(row['slide_name'], row['url'], model,
                                                                  cfg.maxmil_k, cfg.infer_batch)
                if slide_max_prob is None:
                    continue
                val_max_probs.append(slide_max_prob); val_targets.append(row['label'])
            preds = [1 if p >= 0.5 else 0 for p in val_max_probs]
            val_acc = float(np.mean([p == t for p, t in zip(preds, val_targets)])) if val_targets else 0.
            val_loss = 1. - val_acc
            print(f'[Max-MIL] epoch {epoch+1}  val_acc={val_acc:.4f}')
            early_stop(val_loss, model.state_dict(), extra={'epoch': epoch + 1})
            if early_stop.early_stop:
                print('Early stopping Max-MIL training.')
                break
        else:
            torch.save({'state_dict': model.state_dict(), 'epoch': epoch + 1}, cfg.mil_model)

    print('Max-MIL training done. Checkpoint at', cfg.mil_model)
    return model


In [ ]:
# 11. Final feature extraction -- ONE pass per slide using the now-trained Max-MIL encoder,
#     extracting every tissue patch's 512-d pooled feature (not just top-k). Same
#     download -> extract -> delete discipline as before.
def process_one_slide_features(name, url, model, cfg):
    coords = ensure_grid(name, url)
    if not coords:
        return np.zeros((0, 512), dtype=np.float32)
    ext = '.tif' if not url.startswith('gdrive:') else '.svs'
    raw_path = os.path.join(SCRATCH, name + ext)
    try:
        download_slide(url, raw_path)
        slide = openslide.OpenSlide(raw_path)
        feats, buf_imgs = [], []

        def flush():
            if not buf_imgs:
                return
            x = torch.stack(buf_imgs).to(device)
            with torch.no_grad():
                _, f = model(x)
            feats.extend(f.cpu().numpy())
            buf_imgs.clear()

        for coord in coords:
            buf_imgs.append(to_tensor(read_tile(slide, coord, cfg.tile_size, cfg.level)))
            if len(buf_imgs) == cfg.infer_batch:
                flush()
        flush()
        slide.close()
        return np.array(feats, dtype=np.float32)
    finally:
        if os.path.exists(raw_path):
            os.remove(raw_path)


def run_feature_extraction(cfg, model):
    model.eval()
    rows = load_manifest(cfg)
    libs = {'train': {'slides': [], 'grid': [], 'targets': []},
            'val':   {'slides': [], 'grid': [], 'targets': []},
            'test':  {'slides': [], 'grid': [], 'targets': []}}

    for row in rows:
        name, url, label, split = row['slide_name'], row['url'], row['label'], row['split']
        feat_path = os.path.join(cfg.feat_dir, name + cfg.feat_format)
        gp = grid_path_for(name)
        if not (os.path.exists(feat_path) and os.path.exists(gp)):
            feats = process_one_slide_features(name, url, model, cfg)
            np.save(feat_path, feats)
            print(f'[{name}] saved {feats.shape[0]} feature vectors')
        else:
            print(f'[{name}] already extracted, skipping')

        coords = [tuple(map(int, l.strip().split(','))) for l in open(gp)] if os.path.exists(gp) else []
        libs[split]['slides'].append(name)
        libs[split]['targets'].append(label)
        libs[split]['grid'].append(coords)

        for split_name, lib in libs.items():
            torch.save(lib, os.path.join(cfg.lib_dir, f'{split_name}.ckpt'))

    print('lib/{train,val,test}.ckpt written to', cfg.lib_dir)
    return libs


In [ ]:
# 12. Prototype discovery (cluster.py logic) -- unchanged, operates only on the small saved
#     feature files, never on raw images.
from sklearn.cluster import AffinityPropagation
from sklearn.metrics.pairwise import pairwise_distances
import pickle

def euclidean_similarity(x, lamb):
    dists = pairwise_distances(x, metric='euclidean')
    dists = dists / np.max(dists)
    return np.exp(-dists * lamb)

def local_clustering(cfg, lib):
    local_feats, slideIDXs, gridIDXs = [], [], []
    for i, name in enumerate(lib['slides']):
        feat = np.load(os.path.join(cfg.feat_dir, name + cfg.feat_format))
        if feat.shape[0] < 2:
            continue
        sim = euclidean_similarity(feat, cfg.lamb)
        ap = AffinityPropagation(preference=cfg.preference, damping=cfg.damping,
                                  affinity='precomputed', random_state=24).fit(sim)
        centers = ap.cluster_centers_indices_
        slideIDXs.extend([i] * len(centers)); gridIDXs.extend(centers)
        local_feats.append(feat[centers])
        print(f'{name}: {len(centers)} local prototypes')
    local_feats = np.concatenate(local_feats, axis=0) if local_feats else np.zeros((0, 512))
    return local_feats, slideIDXs, gridIDXs

def global_clustering(cfg, local_feats):
    sim = euclidean_similarity(local_feats, cfg.lamb)
    ap = AffinityPropagation(preference=cfg.preference, damping=cfg.damping,
                              affinity='precomputed', random_state=24).fit(sim)
    centers = ap.cluster_centers_indices_
    print('Number of global prototypes:', len(centers))
    return local_feats[centers], ap

def run_clustering(cfg):
    lib = torch.load(os.path.join(cfg.lib_dir, 'train.ckpt'))
    local_feats, slideIDXs, gridIDXs = local_clustering(cfg, lib)
    prototype_feats, ap = global_clustering(cfg, local_feats)
    np.savez(cfg.global_cluster, feature=prototype_feats)
    with open(os.path.join(cfg.cluster_dir, 'ap_model.pkl'), 'wb') as f:
        pickle.dump(ap, f)
    print('Saved prototypes to', cfg.global_cluster)
    return prototype_feats


In [ ]:
# 13. PMIL training & evaluation (pmil.py logic) -- reads only the small saved feature files.
from sklearn.metrics import roc_curve, auc as sk_auc
import torch.utils.data as tdata

class GlobalDataset(tdata.Dataset):
    def __init__(self, feat_dir, libraryfile, feat_format='.npy'):
        lib = torch.load(libraryfile)
        self.slidenames = lib['slides']
        self.targets = lib['targets']   # plural key -- matches what pmil.py/max-mil.py actually read
        self.feats = [np.load(os.path.join(feat_dir, name + feat_format)).astype(np.float32)
                      for name in self.slidenames]

    def __getitem__(self, idx):
        return self.feats[idx], self.targets[idx], self.slidenames[idx]

    def __len__(self):
        return len(self.slidenames)


def pmil_train_epoch(prototype, loader, model, criterion, optimizer):
    model.train()
    running_loss, probs, targets = 0., [], []
    for feats, target, _ in loader:
        feats = torch.as_tensor(feats[0], dtype=torch.float32, device=device)
        if feats.size(0) == 0:
            continue
        target = torch.as_tensor([target[0]], device=device)
        ins_pred, bag_pred, _ = model(feats, prototype)
        score = F.softmax(ins_pred, dim=1)
        max_pred = ins_pred[torch.argmax(score[:, 1]), :]
        loss = criterion(bag_pred.view(1, -1), target) + criterion(max_pred.view(1, -1), target)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        running_loss += loss.item()
        probs.append(F.softmax(bag_pred.detach(), dim=1).squeeze().cpu().numpy()[1])
        targets.append(target.item())
    acc = float(np.mean([(p >= 0.5) == t for p, t in zip(probs, targets)])) if targets else 0.
    return running_loss / max(len(loader.dataset), 1), acc

def pmil_evaluate(prototype, loader, model, criterion):
    model.eval()
    running_loss, probs, targets = 0., [], []
    with torch.no_grad():
        for feats, target, _ in loader:
            feats = torch.as_tensor(feats[0], dtype=torch.float32, device=device)
            if feats.size(0) == 0:
                continue
            target = torch.as_tensor([target[0]], device=device)
            ins_pred, bag_pred, _ = model(feats, prototype)
            max_pred = ins_pred[torch.argmax(F.softmax(ins_pred, dim=1)[:, 1]), :]
            loss = criterion(bag_pred.view(1, -1), target) + criterion(max_pred.view(1, -1), target)
            running_loss += loss.item()
            probs.append(F.softmax(bag_pred.detach(), dim=1).squeeze().cpu().numpy()[1])
            targets.append(target.item())
    if len(set(targets)) < 2:
        roc_auc = float('nan')
    else:
        fpr, tpr, _ = roc_curve(targets, probs, pos_label=1)
        roc_auc = sk_auc(fpr, tpr)
    acc = float(np.mean([(p >= 0.5) == t for p, t in zip(probs, targets)])) if targets else 0.
    return running_loss / max(len(loader.dataset), 1), acc, roc_auc

def run_pmil_training(cfg):
    proto = np.load(cfg.global_cluster)['feature']
    prototype = torch.from_numpy(proto).to(device)
    model = PBMIL(IClassifier(cfg.mil_model), BClassifier(num_cluster=proto.shape[0])).to(device)
    criterion = nn.CrossEntropyLoss().to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
    scheduler = MultiStepLR(optimizer, milestones=[10, 20], gamma=0.1)

    train_loader = tdata.DataLoader(GlobalDataset(cfg.feat_dir, os.path.join(cfg.lib_dir, 'train.ckpt'), cfg.feat_format),
                                     batch_size=1, shuffle=True)
    val_loader = tdata.DataLoader(GlobalDataset(cfg.feat_dir, os.path.join(cfg.lib_dir, 'val.ckpt'), cfg.feat_format),
                                   batch_size=1, shuffle=False)

    # separate "latest" (every epoch, so a disconnect never loses progress) and "best"
    # (only overwritten on improvement) checkpoints
    pmil_latest = cfg.pmil_model.replace('.pth', '_latest.pth')
    early_stop = EarlyStopping(cfg.pmil_model, patience=7, verbose=True)  # writes the "best" checkpoint
    best_acc = 0.
    start_epoch = 0
    if os.path.exists(pmil_latest):
        ch = torch.load(pmil_latest, map_location='cpu')
        model.load_state_dict(ch['state_dict'])
        start_epoch = ch.get('epoch', 0)
        best_acc = ch.get('best_acc', 0.)
        print(f'Resuming PMIL training from epoch {start_epoch} (latest checkpoint found)')

    for epoch in range(start_epoch, cfg.nepochs):
        train_loss, train_acc = pmil_train_epoch(prototype, train_loader, model, criterion, optimizer)
        val_loss, val_acc, val_auc = pmil_evaluate(prototype, val_loader, model, criterion)
        scheduler.step()
        print(f'Epoch {epoch+1}/{cfg.nepochs}  train_loss={train_loss:.4f} acc={train_acc:.4f} | '
              f'val_loss={val_loss:.4f} acc={val_acc:.4f} auc={val_auc:.4f}')
        if val_acc >= best_acc:
            best_acc = val_acc
        torch.save({'state_dict': model.state_dict(), 'epoch': epoch + 1, 'best_acc': best_acc}, pmil_latest)
        early_stop(val_loss, model.state_dict(), extra={'best_acc': best_acc})
        if early_stop.early_stop:
            print('Early stopping'); break
    return model, prototype

def run_pmil_test(cfg, model=None, prototype=None):
    if model is None:
        proto = np.load(cfg.global_cluster)['feature']
        prototype = torch.from_numpy(proto).to(device)
        model = PBMIL(IClassifier(cfg.mil_model), BClassifier(num_cluster=proto.shape[0])).to(device)
        ch = torch.load(cfg.pmil_model, map_location='cpu')
        model.load_state_dict(ch['state_dict'])
    test_loader = tdata.DataLoader(GlobalDataset(cfg.feat_dir, os.path.join(cfg.lib_dir, 'test.ckpt'), cfg.feat_format),
                                    batch_size=1, shuffle=False)
    criterion = nn.CrossEntropyLoss().to(device)
    loss, acc, auc_val = pmil_evaluate(prototype, test_loader, model, criterion)
    print(f'TEST  loss={loss:.4f}  acc={acc:.4f}  auc={auc_val:.4f}')
    return loss, acc, auc_val


## Run it -- staged, don't jump straight to the full manifest

**Stage 1 (1 slide):** put a single row in `manifest.csv`, run cells 14-15, then inspect
`feat/<name>.npy` shape (`(N, 512)`) and eyeball a couple of tiles.

**Stage 2 (5 slides):** add a few more rows (mix of labels/splits), rerun. Confirm it resumes
correctly if interrupted (re-running skips slides whose `.npy` AND `.grid.csv` both already exist).

**Stage 3 (full manifest):** only after 1 and 2 work. Expect Max-MIL training to be the slow part —
it downloads every slide once per `maxmil_nepochs`. Start with `maxmil_nepochs = 3-5`.


In [ ]:
# 14. Stage: Max-MIL training (streaming, storage-frugal, re-scores all patches every epoch)
maxmil_model = run_maxmil_training(cfg)


In [ ]:
# 15. Stage: final feature extraction with the trained Max-MIL encoder
run_feature_extraction(cfg, maxmil_model)


In [ ]:
# 16. Stage: prototype discovery
run_clustering(cfg)


In [ ]:
# 17. Stage: PMIL training
pmil_model, prototype = run_pmil_training(cfg)


In [ ]:
# 18. Stage: PMIL testing
run_pmil_test(cfg, pmil_model, prototype)


### Remaining honest caveats

- Tiling defaults (`tile_size`, `level`, `tissue_thresh`) aren't tuned to Camelyon16 specifically —
  check tiles visually before scaling up.
- `maxmil_nepochs` re-downloads the entire manifest that many times. If your slides are hosted
  somewhere slow, this dominates runtime — consider a smaller training subset for Max-MIL, then
  extracting features for the full manifest afterward (stage 15 only needs the trained checkpoint,
  not the same slide list used in stage 14).
- Early stopping / checkpointing happens at epoch boundaries in both the Max-MIL and PMIL stages —
  if Colab disconnects mid-epoch you'll redo that epoch, not the whole run, since the last saved
  checkpoint in `model/` is reloaded by re-running the corresponding stage cell (add a small
  `if os.path.exists(cfg.mil_model): load it before continuing` guard at the top of stage 14 if you
  want epoch-level resume rather than restarting Max-MIL from ImageNet weights).
